# Anime 데이터 Migration

In [ ]:
import dotenv
import os
import json
from pymongo import MongoClient
from datetime import datetime
import psycopg2
from psycopg2.extras import execute_batch

In [ ]:
dotenv.load_dotenv(dotenv.find_dotenv())
USER = os.environ["MONGODB_USER"] # MongoDB user
PASSWORD = os.environ["MONGODB_PASSWORD"] # MongoDB password
HOST = os.environ["MONGODB_HOST"]
PORT = int(os.environ["MONGODB_PORT"]) # MongoDB port

# DB 연결
conn = MongoClient("mongodb://" + HOST + ":" + str(PORT))
db = conn.chuanione

# collection 불러오기
dbcol_detail = db.ani_detail
dbcol_reviews = db.reviews

pg = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=os.environ["POSTGRES_PORT"],
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
)
cur = pg.cursor()


In [ ]:
# Anime 문서 가져오기
docs = dbcol_detail.find()

# anime 테이블 데이터 준비
anime_rows = []
metadata_rows = []

for doc in docs:
    # name이 없거나 None이면 pass
    if not doc.get("name"):
        continue

    anime_rows.append((
        doc["id"],
        doc.get("name"),
        doc.get("series_id"),
        doc.get("air_year_quarter"),
        doc.get("medium"),
        doc.get("content"),  # synopsis
        doc.get("production"),
        doc.get("avg_rating"),
        doc.get("is_adult"),
        doc.get("is_dubbed"),
        doc.get("is_uncensored"),
        datetime.fromisoformat(doc["latest_episode_release_datetime"]) if doc.get("latest_episode_release_datetime") else None,
        datetime.now(),
        datetime.now()
    ))

    metadata_rows.append((
        doc["id"],
        json.dumps(doc.get("casts")),
        json.dumps(doc.get("tags")),
        json.dumps(doc.get("genre")),
        json.dumps(doc.get("images")),
        json.dumps(doc.get("directors")),
        json.dumps(doc.get("production_companies")),
        json.dumps(doc.get("awards")),
        json.dumps(doc.get("highlight_video")) if doc.get("highlight_video") else None
    ))

# anime 테이블에 배치 insert
execute_batch(cur, """
    INSERT INTO anime (id, name, series_id, air_year_quarter, medium, synopsis, production,
                       avg_rating, is_adult, is_dubbed, is_uncensored,
                       latest_episode_release_datetime, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s)
    ON CONFLICT (id) DO NOTHING;
""", anime_rows, page_size=1000)

# anime_metadata 테이블에 배치 insert
execute_batch(cur, """
    INSERT INTO anime_metadata (anime_id, casts, tags, genres, images, directors,
                                production_companies, awards, highlight_video)
    VALUES (%s, %s, %s, %s, %s, %s,
            %s, %s, %s)
    ON CONFLICT (anime_id) DO NOTHING;
""", metadata_rows, page_size=1000)


In [ ]:
reviews = list(dbcol_reviews.find())

review_rows = []

for r in reviews:
    # 필수값 체크: anime_id와 rating이 없으면 pass
    if not r.get("ani_id") or r.get("score") is None:
        continue

    review_rows.append((
        r.get("ani_id"),
        r.get("user_id"),
        r.get("score"),
        r.get("content"),
        r.get("like_count"),
        datetime.fromisoformat(r["created_at"]) if r.get("created_at") else datetime.now(),
        datetime.fromisoformat(r["updated_at"]) if r.get("updated_at") else datetime.now()
    ))

# 리뷰 데이터 배치 insert
execute_batch(cur, """
    INSERT INTO anime_reviews (anime_id, user_id, score, content, like_count, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT DO NOTHING;
""", review_rows, page_size=1000)



In [27]:
pg.commit()
cur.close()
pg.close()